In [1]:
import pandas as pd
import numpy as np

An untouched hurdle here is selection of continious, categorical(counts), means, or proportions to use on the axes opposite proportion, mean, count, or continuious data from another variable. There are many ways to do it, and many different use cases. Also, there can be one singe variable instead of two. Such as cases where CDF is used.   

Morever, error bars for means and propotions could make the plots multi functional, such as **pareto or non-pareto** with or without multiple axes.   

# consider two(or3) types:  
>regular pareto [categorical counts on left, proportions on right]   
>weighted pareto [categorical on left, (sums of means or sums of means*k) on right]  < -- 1 or 2 types here  
and options for a) no right axis, b)no error bars on bars, c) no error bars on line  

also: add default cross hairs at 80% and 20% for the true pareto measure  

it should be it's own class and inherit from MuEstimator   

there should be an approach to put a categorical on the x and y axis, such as acumulative count or proportion of single or grouped subcats from a dependent "target" column

In [ ]:
#these two df priducing functions where derived from get_mean_estimate_df() and get_proportion_estimate_df() in ~/Consumer_Habits.ipynb

def get_proportion_MOE_df(dataframe:pd.DataFrame,target_col:str,confidence_level:float=0.95,partition_by:list=None):
    """
    takes a dataframe, target column, confidence interval[default 0.95], optional partition column(s)[default None], and sort[default None] as input.
    returns a new dataframe with <partition column>, target column, num_observations column, upper column, and lower column.
    where upper and lower are the mu estimate intervals
    if sort is not None, it should be one of True or False to indicate ascending=True/False proportions
    """
    if partition_by is None or partition_by == []:
        estimate_df=get_single_variable_proportions(dataframe[target_col])
    else:
        estimate_df=get_bivariable_proportions(dataframe, partition_by, target_col)
    if confidence_level>0.999999:
        raise ValueError('Confidence Level Out of Bounds\nexceeds-->0.999999')
    MOE=MuEstimator().proportion_estimator(estimate_df['proportion'],estimate_df['num_observations'],confidence_level,)
    estimate_df['error']=MOE*estimate_df['num_observations']
    if partition_by!=None or partition_by!=[]:
        estimate_df=estimate_df.sort_values(by=[target_col]+['successes','proportion']+partition_by,ascending=[False,True,True]+[False for i in partition_by]).reset_index(drop=True)
    else: 
        estimate_df=estimate_df.sort_values(by=[target_col,'successes','proportion'],ascending=[False,True,True]).reset_index(drop=True)        
    return estimate_df



def get_mean_MOE_df(df,target_col:str,confidence_level:float=0.95,partition_cols:list=None):
    """
    takes dataframe, a target column, a confidence interval[default 0.95], a list of 0 to n partition columns to group by, and sort[default None].
    return a grouped dataframe with aggregated columns: 'min','mean','median','max','std','size','lower','upper'
    if sort is not None, it should be one of True or False to indicate ascending=True/False for sort by 'mean'
    """
    if partition_cols==None or partition_cols==[]:
        estimate_df = pd.DataFrame({target_col:[target_col],'mean':[df[target_col].mean()],'median':[df[target_col].median()],'std':[df[target_col].std()],'size':[df[target_col].count()]})
    else:
        estimate_df = df.groupby(partition_cols,as_index=False,observed=True)[target_col].agg(['mean','std','size']) 
    MOE=MuEstimator().mean_estimator(estimate_df['mean'],estimate_df['std'],estimate_df['size'],confidence_level,MOE_only=True)
    estimate_df['error']=MOE
    estimate_df=estimate_df.drop(columns='std')
    if partition_cols!=None or partition_cols!=[]:
        estimate_df=estimate_df.sort_values(by=['size','mean']+partition_cols,ascending=[True,True]+[False for i in partition_cols]).reset_index(drop=True)
    else:
        estimate_df=estimate_df.sort_values(by=['size','mean'],ascending=[True,True]).reset_index(drop=True)
    return estimate_df

#get plot data was derived from plot_floating_mu_hbar()
def get_error_bar_plot_data(plot_df_:pd.DataFrame,target_col:str,partition_cols:list):
    """
    return lefts,mu,median,widths,X_ticks,Y_label,intersects,target_col
    where lefts is lower bounds, mu is mu, media is median or none in case of proportions, widths is MOE*2, 
    X_ticks is concatenated partition feature labels, 
    Y_label is target col w/ partitions by feature label, 
    intersects is a list of feature partitions, 
    target col is the string title of the target col that represents the mu
    """
    
    plot_df=plot_df_.copy()
    if target_col not in plot_df.columns:
        plot_df[target_col]=target_col
    '''plot_df=plot_df.sort_values(by=partition_cols+[target_col,'size'],ascending=[True for i in partition_cols+[target_col]]+[False]).reset_index(drop=True)
    else:
        plot_df=plot_df_.copy().sort_values(by=partition_cols+[target_col,'successes'],ascending=[True for i in partition_cols+[target_col]]+[False]).reset_index(drop=True)'''
    partitions=False if partition_cols is None or partition_cols==[] else True
    if partitions==True:
        X_ticks="] -> "+plot_df[target_col].astype(str)
    else: X_ticks=plot_df[target_col].astype(str)
    first=True  # to help with string format
    for col in partition_cols: 
        if first==False:
            X_ticks="'"+plot_df[col].astype(str)+"', "+X_ticks.astype(str)
        else:
            X_ticks="'"+plot_df[col].astype(str)+"'"+X_ticks.astype(str)
            first=False
    if partitions==True:
            X_ticks="["+X_ticks.astype(str)
    X_ticks=X_ticks.values  
    #Y_label    
    Y_label=target_col+' Partitioned By '+str(partition_cols) if partitions==True else target_col.title()
    #check categorical or continuous
    mu_is_mean=True if 'mean' in plot_df_.columns else False     
    if mu_is_mean==True: 
        mu,counts,MOE,median =  plot_df['mean'].values, plot_df['size'].values, plot_df['error'].values, plot_df['median'].values
    else:
        mu,counts,MOE=plot_df['proportion'].values, plot_df['successes'].values, plot_df['error'].values
        median=None
    intersects = partition_cols if partition_cols is not None else []
    return mu,median,MOE,X_ticks,Y_label,intersects,counts,target_col

def bar_w_errors(mu,median,MOE,X_ticks,Y_label,intersects,counts,target_col):
    pass


plt.bar(
    x= None,
    height=None,
    yerr=None,
    capsize=None,
    )

**Matplotlib:** ❌ **No true Pareto plot out of the box**
**Seaborn:** ❌ **No Pareto plot at all**

But it’s easy to build one using standard pieces.

---

## What a Pareto plot actually is

A Pareto chart =

1. **Bars** (usually counts, sorted descending)
2. **Cumulative percentage line** on a **secondary y-axis**

That second axis is why there’s no single built-in call.

---

## Canonical Matplotlib Pareto (10 lines)

```python
import numpy as np
import matplotlib.pyplot as plt

counts = np.array([40, 25, 20, 10, 5])
labels = ["A", "B", "C", "D", "E"]

# sort
idx = np.argsort(counts)[::-1]
counts = counts[idx]
labels = np.array(labels)[idx]

cumperc = counts.cumsum() / counts.sum() * 100

fig, ax = plt.subplots()
ax.bar(labels, counts)
ax.set_ylabel("Count")

ax2 = ax.twinx()
ax2.plot(labels, cumperc, marker="o")
ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0, 100)

plt.show()
```

---

## Seaborn-style Pareto (still Matplotlib under the hood)

```python
sns.barplot(x=labels, y=counts, ax=ax)
ax2 = ax.twinx()
ax2.plot(labels, cumperc, marker="o")
```

Seaborn **does not manage** the cumulative axis for you.

---

## Libraries that *do* have Pareto helpers

* **statsmodels** → ❌ no
* **scipy** → ❌ no
* **plotly** → ❌ no single call (same manual steps)
* **pandas** → ❌ (but `value_counts().cumsum()` helps)

So everyone builds it manually.

---

## Why this is intentional

Pareto charts:

* require sorting
* require cumulative normalization
* require a secondary axis
  → too opinionated for a generic plotting API

---

## Recommendation

Create a **reusable function** once and forget about it.
If you want, I can give you:

* a seaborn-styled reusable `pareto_plot(df, col)`
* a Streamlit-safe version
* a version with confidence intervals or proportions

Just say which.


Use a **secondary y-axis** on the *same x-axis*. That’s exactly what `twinx()` is for.

### Minimal pattern (Matplotlib)

```python
fig, ax1 = plt.subplots()

# first plot (left y-axis)
ax1.plot(x, y1, label="Series A")
ax1.set_ylabel("Scale A")

# second plot (right y-axis)
ax2 = ax1.twinx()
ax2.plot(x, y2, label="Series B")
ax2.set_ylabel("Scale B")

# optional: one combined legend
lines = ax1.lines + ax2.lines
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels)

plt.show()
```

### Key points

* **Same axis space**, same x-values
* **Left y-axis** = `ax1`
* **Right y-axis** = `ax2 = ax1.twinx()`
* Each axis has its **own scale, ticks, and label**

### With Seaborn

Plot on explicit axes:

```python
sns.lineplot(x=x, y=y1, ax=ax1)
sns.lineplot(x=x, y=y2, ax=ax2)
```

### When *not* to do this

* If the relationship between scales isn’t meaningful → viewers can misinterpret
* If scales differ wildly → consider normalization or small multiples instead

If you want:

* different colors synced to axis sides
* log scale on one side only
* bars on one axis + lines on the other
  say the word and I’ll show the exact pattern.
